# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShayanAhmed12/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 4 — CTR / Engagement Opportunity Scoring** (core lane, starting on the starter dataset, moving to the warehouse release in Week 3).

Why this one: in notebook 01 I saw that click-through rate collapses with position (Discovery B) and that, *within the same position tier*, CTR still varies a lot between pages (my "your turn" cell). That leftover spread is the interesting part: two pages sitting at the same rank with the same exposure, where one gets far fewer clicks than its neighbours. Those pages are already visible, so the opportunity is to win clicks from impressions they already have, rather than to win new rankings. The numbers in section 3 suggest this is not a niche: most impressions in the sample sit on pages 1–2 of results, and a sizeable group of those pages is clearly under-clicking for its tier.


In [1]:
# Setup: load the starter dataset (runs locally or in Colab)
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("data/raw") and os.path.dirname(os.getcwd()) != os.getcwd():
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"{len(df):,} pages, {df['client_id'].nunique()} clients, {df.shape[1]} columns")


30,000 pages, 32 clients, 44 columns


## 2. The question: decision, action, cost of a wrong call

**Search question:** *Among pages that are already visible in Google (enough impressions, average position 1–20), which ones get clearly fewer clicks than pages at a similar position — and should be reviewed first?*

- **Unit of analysis:** one content page (one row = one page with its trailing 90-day metrics). Later on the warehouse: one page over a fixed feature window.
- **Output:** a **ranked review queue** of pages, each with an *expected CTR for its position*, its *CTR gap* (actual vs expected), the *clicks at stake* (gap × impressions), and a short **reason code** (e.g. `high_impressions_low_ctr`, `strong_position_low_ctr`, `low_volume_monitor`).
- **Decision it improves:** *which visible pages should an SEO/content editor look at first this week?* Editors have limited hours; today they would pick by gut or by raw CTR, which unfairly flags every low-ranked page.
- **Who acts and how:** the content/SEO editor opens the top of the queue and reviews the search snippet — rewrites the title and meta description, checks the page actually matches the search intent, or marks it "monitor" if the gap looks like noise or a SERP feature (e.g. an answer box) they can't influence.
- **Cost of a wrong call:**
  - *False positive* (page flagged but nothing is wrong): ~30–60 min of wasted editor time, and a risk of rewriting a title that was actually fine.
  - *False negative* (an under-clicking page is missed): clicks keep leaking every day on impressions the page already earns. On the busiest pages this is the costlier error, so the queue should weight by clicks at stake, not by CTR gap alone.
- **Why data / ML helps at all:** "low CTR" by itself is meaningless — CTR depends heavily on position, and also on content type, intent and volume (small pages have very noisy CTR). A single fixed threshold like `ctr < 0.5` mixes all of these up. Data lets us build a fair **expected CTR** for each page from comparable pages, and a model can later combine several context signals at once. This is **not "just train a model"**: the first, most important step is a transparent position-adjusted baseline (tier median), and a model only earns its place if it ranks the queue better than that baseline on clients it has never seen.


In [2]:
# The framing in one paragraph (the template from skills/framing-ml-problems)
frame = (
    "For an SEO/content editor, deciding which visible pages to review first, "
    "we will build a ranked review queue from page-level search metrics, "
    "scoring the gap between a page's CTR and the expected CTR for its position, "
    "measured by precision@K of the queue against a held-out outcome. "
    "A wrong call costs editor time (false positive) or leaking clicks (false negative). "
    "A plain CTR threshold isn't enough because CTR depends on position, volume and content type. "
    "We will claim only observed / directional / decision-support results."
)
print(frame)


For an SEO/content editor, deciding which visible pages to review first, we will build a ranked review queue from page-level search metrics, scoring the gap between a page's CTR and the expected CTR for its position, measured by precision@K of the queue against a held-out outcome. A wrong call costs editor time (false positive) or leaking clicks (false negative). A plain CTR threshold isn't enough because CTR depends on position, volume and content type. We will claim only observed / directional / decision-support results.


## 3. Quick look at the data (2-3 real numbers)

Below I compute, live from the starter CSV: (1) how much of all search exposure sits on pages that are already visible (position 1–20, ≥500 impressions in 90 days), (2) how widely CTR spreads *within* the same position tier, and (3) how many visible pages are under-clicking badly — and roughly how many clicks are at stake.


In [3]:
# Notes: ctr is a x100 percentage (0.24 = 0.24%). avg_position == 0 means "no data", so it is excluded.
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
share_impr = visible["impressions_90d"].sum() / df["impressions_90d"].sum()
print(f"1) Visible pages (pos 1-20, >=500 impressions): {len(visible):,} of {len(df):,} pages "
      f"({len(visible)/len(df):.0%}) from {visible['client_id'].nunique()} clients, "
      f"but they hold {share_impr:.0%} of all impressions.")

# 2) Spread of CTR inside each position tier (tiers with >=100 pages)
tiers = visible.groupby("position_tier")["ctr"]
spread = tiers.describe()[["count", "25%", "50%", "75%"]]
spread = spread[spread["count"] >= 100].sort_values("50%", ascending=False)
print("\n2) CTR (%) inside each tier - same position, very different click rates:")
print(spread.round(3).to_string())
p1 = visible[visible["position_tier"] == "page_1"]["ctr"]
print(f"   On page_1, the top-quartile page gets {p1.quantile(.75)/p1.quantile(.25):.1f}x the CTR of the bottom-quartile page.")

# 3) Pages earning less than half their tier's median CTR, and the clicks they leave on the table
visible["expected_ctr"] = visible.groupby("position_tier")["ctr"].transform("median")
under = visible[visible["ctr"] < 0.5 * visible["expected_ctr"]]
clicks_at_stake = ((under["expected_ctr"] - under["ctr"]) / 100 * under["impressions_90d"]).sum()
print(f"\n3) {len(under):,} visible pages ({len(under)/len(visible):.0%}) get under half the median CTR of their tier.")
print(f"   They currently earn {int(under['clicks_90d'].sum()):,} clicks in 90 days; reaching just the tier median "
      f"would mean roughly {int(clicks_at_stake):,} more (~{clicks_at_stake/under['clicks_90d'].sum():.1f}x).")


1) Visible pages (pos 1-20, >=500 impressions): 12,023 of 30,000 pages (40%) from 28 clients, but they hold 76% of all impressions.

2) CTR (%) inside each tier - same position, very different click rates:
                count   25%   50%   75%
position_tier                          
page_1         7064.0  0.12  0.24  0.44
top_3           458.0  0.08  0.20  0.49
striking       4485.0  0.08  0.17  0.34
   On page_1, the top-quartile page gets 3.7x the CTR of the bottom-quartile page.

3) 3,031 visible pages (25%) get under half the median CTR of their tier.
   They currently earn 13,135 clicks in 90 days; reaching just the tier median would mean roughly 35,452 more (~2.7x).


## 4. Careful words: what I can and can't claim

**What this work will be able to say:**
- *Observed:* "In this data, page X gets N% of the clicks that pages at a similar position typically get."
- *Directional:* "Pages with these characteristics tend to under-capture clicks for their position."
- *Decision-support:* "If an editor can review 50 pages this week, these 50 are the most promising places to look first" — and, once validated on held-out clients, "this queue finds more real under-performers in its top 50 than a simple CTR rule does."

**What it will never say:**
- That rewriting a title **caused** more clicks — this is observational data; proving cause would need an experiment (e.g. a before/after test with a control group).
- That I know how **Google ranks or displays** pages — I see outcomes (impressions, clicks, position), not the algorithm or the SERP features (answer boxes, ads, AI overviews) that can also steal clicks.
- That a low-CTR page **is bad** — a gap can come from search intent, brand queries, SERP layout or plain noise; the queue says "review this", not "this is broken".
- Anything beyond this sample: these numbers come from a 30k-page anonymized slice (28 clients after filtering); the "clicks at stake" figure is a rough upper-bound illustration, not a forecast. Everything must be re-checked on the warehouse release.

**Known traps I'll guard against:** comparing CTR across positions without adjusting; trusting CTR on tiny impression counts; leaking `clicks`/`ctr` into a label built from the same window; letting one big client dominate (the comparison-article finding in notebook 01 came from a single client).

**What the check below shows:** CTR on small pages is ~20x noisier than on visible pages, so the ≥500-impression floor is needed. And one client accounts for **43%** of the under-clicking pages, so every result must be reported per client and validated on held-out clients, not just pooled.


In [4]:
# Sanity checks behind the claims above
# a) volume floor matters: CTR noise on small pages vs visible pages
small = df[(df["impressions_90d"] > 0) & (df["impressions_90d"] < 100) & (df["avg_position"].between(0.01, 20))]
print(f"CTR std on small pages (<100 impressions): {small['ctr'].std():.2f}  vs visible pages: {visible['ctr'].std():.2f}")
# b) concentration: does one client dominate the under-clicking group?
top_share = under["client_id"].value_counts(normalize=True).iloc[0]
print(f"Largest single client's share of the under-clicking pages: {top_share:.0%} "
      f"({under['client_id'].nunique()} clients represented)")


CTR std on small pages (<100 impressions): 7.00  vs visible pages: 0.34
Largest single client's share of the under-clicking pages: 43% (22 clients represented)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.